<a href="https://colab.research.google.com/github/leonmarienga/ML-thesis/blob/main/reinforcement_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install stable-baselines3 gymnasium pandas numpy matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 16.0 MB/s eta 0:00:00
  Attempting uninstall: gymnasium
    Found existing installation: gymnasium 1.3.0
    Uninstalling gymnasium-1.3.0:
      Successfully uninstalled gymnasium-1.3.0


In [2]:
from google.colab import files

uploaded = files.upload()

Saving rl_training_data.csv to rl_training_data.csv


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym

from gymnasium import spaces
from collections import Counter

from stable_baselines3 import DQN, PPO, A2C
from stable_baselines3.common.env_checker import check_env

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [4]:
RL_TRAINING_DATA_PATH = "rl_training_data.csv"

TOTAL_BUDGET = 8_000_000_000
INITIAL_RELEASED_BUDGET = 900_000_000

SCENARIOS_PER_PAGE = 5

FUNDING_RELEASE_SCHEDULE = {
    1: 900_000_000,
    15: 700_000_000,
    32: 1_100_000_000,
    54: 1_300_000_000,
    79: 850_000_000,
    108: 1_000_000_000,
    136: 950_000_000,
    160: 1_200_000_000,
}

ACTION_MULTIPLIERS = [
    0.40,
    0.55,
    0.70,
    0.85,
    1.00,
    1.10,
]

RANDOM_SEED = 42

In [5]:
df = pd.read_csv(RL_TRAINING_DATA_PATH)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

df[["ml_base_prediction", "true_funding"]].describe()

Rows: 971
Columns: ['state', 'incidentType', 'expectedResourceLevel', 'disasterCategory', 'durationClass', 'durationDays', 'declarationDelayDays', 'fyDeclared', 'ihProgramDeclared', 'paProgramDeclared', 'hmProgramDeclared', 'expectedResourceScore', 'missionAssignmentCount', 'uniqueAgencyCount', 'uniqueMaTypeCount', 'uniquePriorityCount', 'responseComplexityScore', 'missionDensity', 'agencyDensity', 'stage1_funded_prediction', 'ml_base_prediction', 'true_funding', 'totalObligatedFunding']


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,ml_base_prediction,true_funding
count,9.710000e+02,9.710000e+02
mean,1.789355e+07,2.097639e+07
std,1.614430e+08,1.904758e+08
min,0.000000e+00,0.000000e+00
25%,0.000000e+00,0.000000e+00
50%,0.000000e+00,0.000000e+00
75%,1.430921e+05,1.590370e+05
max,3.998207e+09,4.379126e+09


In [6]:
def calculate_reward(
    final_recommendation,
    true_funding,
    remaining_budget_after,
    total_budget,
    released_budget_so_far,
    current_step,
    total_steps,
    done,
):
    """
    Reward function for budget-release simulation.

    The RL model is rewarded for:
    - staying close to historical funding pattern
    - avoiding severe underfunding
    - not overspending released budget
    - using released budget reasonably
    """

    final_recommendation = max(float(final_recommendation), 0.0)
    true_funding = max(float(true_funding), 0.0)
    remaining_budget_after = float(remaining_budget_after)
    total_budget = max(float(total_budget), 1.0)
    released_budget_so_far = max(float(released_budget_so_far), 1.0)

    log_final = np.log1p(final_recommendation)
    log_true = np.log1p(true_funding)

    log_error = abs(log_final - log_true)

    reward = -log_error

    if remaining_budget_after < 0:
        overspend_ratio = abs(remaining_budget_after) / released_budget_so_far
        reward -= 10.0
        reward -= 20.0 * overspend_ratio

    if true_funding > 0 and final_recommendation < 0.25 * true_funding:
        reward -= 1.5

    remaining_released_ratio = remaining_budget_after / released_budget_so_far
    spent_released_ratio = 1.0 - remaining_released_ratio

    if spent_released_ratio > 0.95:
        reward -= 0.5

    if done and remaining_budget_after > 0:
        unused_ratio = remaining_budget_after / released_budget_so_far
        reward -= 0.25 * unused_ratio

    return float(reward)

In [7]:
class CleanBudgetAdjustmentEnv(gym.Env):
    """
    Clean RL environment for post-ML budget adjustment.

    ML predicts the base funding amount.
    RL only sees budget-pressure information and chooses a multiplier.

    RL does NOT see:
    - page progress
    - total budget progress
    - scenario complexity
    - resource score
    - mission count
    - duration
    - density values
    """

    metadata = {"render_modes": []}

    def __init__(
        self,
        data_path,
        total_budget=TOTAL_BUDGET,
        initial_released_budget=INITIAL_RELEASED_BUDGET,
        funding_release_schedule=FUNDING_RELEASE_SCHEDULE,
        scenarios_per_page=SCENARIOS_PER_PAGE,
        action_multipliers=ACTION_MULTIPLIERS,
        episode_length=200,
        random_seed=42,
    ):
        super().__init__()

        self.df = pd.read_csv(data_path)

        self.total_budget = float(total_budget)
        self.initial_released_budget = float(initial_released_budget)
        self.funding_release_schedule = funding_release_schedule
        self.scenarios_per_page = scenarios_per_page
        self.action_multipliers = action_multipliers
        self.episode_length = min(episode_length, len(self.df))

        self.random_seed = random_seed
        self.rng = np.random.default_rng(random_seed)

        self.action_space = spaces.Discrete(len(self.action_multipliers))

        # Observation vector:
        # 0 log ML base recommendation
        # 1 remaining released budget ratio
        # 2 released budget amount scaled
        # 3 spent released budget ratio
        # 4 ML recommendation / remaining released budget
        # 5 ML recommendation / released budget so far
        self.observation_space = spaces.Box(
            low=-10.0,
            high=10.0,
            shape=(6,),
            dtype=np.float32,
        )

        self.session_df = None
        self.current_step = 0
        self.current_page = 1
        self.released_budget_so_far = self.initial_released_budget
        self.remaining_budget = self.initial_released_budget
        self.applied_release_pages = set()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        if seed is not None:
            self.rng = np.random.default_rng(seed)

        self.session_df = (
            self.df.sample(
                n=self.episode_length,
                replace=False,
                random_state=int(self.rng.integers(0, 1_000_000)),
            )
            .reset_index(drop=True)
        )

        self.current_step = 0
        self.current_page = 1
        self.released_budget_so_far = self.initial_released_budget
        self.remaining_budget = self.initial_released_budget
        self.applied_release_pages = set()
        self.applied_release_pages.add(1)

        observation = self._get_observation()
        info = {}

        return observation, info

    def step(self, action):
        self._apply_funding_release_if_needed()

        row = self.session_df.iloc[self.current_step]

        multiplier = self.action_multipliers[int(action)]

        ml_base = float(row["ml_base_prediction"])
        true_funding = float(row["true_funding"])

        final_recommendation = ml_base * multiplier

        final_recommendation = min(final_recommendation, max(self.remaining_budget, 0.0))
        final_recommendation = max(final_recommendation, 0.0)

        remaining_budget_after = self.remaining_budget - final_recommendation

        done = self.current_step >= self.episode_length - 1

        reward = calculate_reward(
            final_recommendation=final_recommendation,
            true_funding=true_funding,
            remaining_budget_after=remaining_budget_after,
            total_budget=self.total_budget,
            released_budget_so_far=self.released_budget_so_far,
            current_step=self.current_step,
            total_steps=self.episode_length,
            done=done,
        )

        self.remaining_budget = remaining_budget_after
        self.current_step += 1
        self.current_page = (self.current_step // self.scenarios_per_page) + 1

        if done:
            observation = np.zeros(self.observation_space.shape, dtype=np.float32)
        else:
            observation = self._get_observation()

        info = {
            "ml_base_prediction": ml_base,
            "true_funding": true_funding,
            "multiplier": multiplier,
            "final_recommendation": final_recommendation,
            "remaining_budget": self.remaining_budget,
            "released_budget_so_far": self.released_budget_so_far,
            "current_page": self.current_page,
        }

        truncated = False

        return observation, reward, done, truncated, info

    def _apply_funding_release_if_needed(self):
        if self.current_page in self.funding_release_schedule:
            if self.current_page not in self.applied_release_pages:
                release_amount = float(self.funding_release_schedule[self.current_page])

                max_remaining_release = self.total_budget - self.released_budget_so_far
                release_amount = min(release_amount, max_remaining_release)

                self.released_budget_so_far += release_amount
                self.remaining_budget += release_amount

                self.applied_release_pages.add(self.current_page)

    def _get_observation(self):
        row = self.session_df.iloc[self.current_step]

        ml_base = float(row["ml_base_prediction"])

        remaining_budget = max(float(self.remaining_budget), 0.0)
        released_budget = max(float(self.released_budget_so_far), 1.0)

        spent_budget = released_budget - remaining_budget

        remaining_released_ratio = remaining_budget / released_budget
        spent_released_ratio = spent_budget / released_budget

        ml_to_remaining_ratio = ml_base / max(remaining_budget, 1.0)
        ml_to_released_ratio = ml_base / released_budget

        obs = np.array(
            [
                np.log1p(ml_base) / 25.0,
                remaining_released_ratio,
                released_budget / TOTAL_BUDGET,
                spent_released_ratio,
                min(ml_to_remaining_ratio, 10.0),
                min(ml_to_released_ratio, 10.0),
            ],
            dtype=np.float32,
        )

        obs = np.nan_to_num(obs, nan=0.0, posinf=10.0, neginf=-10.0)

        return obs

In [8]:
clean_env = CleanBudgetAdjustmentEnv(
    data_path=RL_TRAINING_DATA_PATH,
    episode_length=200,
    random_seed=RANDOM_SEED,
)

check_env(clean_env, warn=True)

obs, info = clean_env.reset()

print("Observation:", obs)
print("Observation shape:", obs.shape)
print("Action space:", clean_env.action_space)

total_reward = 0

for step in range(20):
    action = clean_env.action_space.sample()
    obs, reward, done, truncated, info = clean_env.step(action)
    total_reward += reward

print("Test reward:", total_reward)
print("Last info:", info)

Observation: [0.7433589  1.         0.1125     0.         0.13082007 0.13082007]
Observation shape: (6,)
Action space: Discrete(6)
Test reward: -2.586499623420057
Last info: {'ml_base_prediction': 0.0, 'true_funding': 0.0, 'multiplier': 0.55, 'final_recommendation': 0.0, 'remaining_budget': 758051660.49, 'released_budget_so_far': 900000000.0, 'current_page': 5}


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [9]:
def choose_rule_based_action(obs):
    remaining_budget_ratio = obs[1]

    if remaining_budget_ratio < 0.20:
        multiplier = 0.55
    elif remaining_budget_ratio < 0.40:
        multiplier = 0.70
    elif remaining_budget_ratio < 0.60:
        multiplier = 0.85
    else:
        multiplier = 1.00

    return ACTION_MULTIPLIERS.index(multiplier)


def choose_ml_only_action():
    return ACTION_MULTIPLIERS.index(1.00)


def run_episode(env, policy_name, model=None):
    obs, _ = env.reset()

    total_reward = 0.0
    total_final_recommendation = 0.0
    total_true_funding = 0.0
    absolute_error = 0.0
    log_absolute_error = 0.0
    overspend_count = 0
    severe_underfund_count = 0
    multipliers = []

    while True:
        if policy_name == "random":
            action = env.action_space.sample()
        elif policy_name == "ml_only":
            action = choose_ml_only_action()
        elif policy_name == "rule_based":
            action = choose_rule_based_action(obs)
        elif policy_name in ["dqn", "ppo", "a2c"]:
            action, _ = model.predict(obs, deterministic=True)
            action = int(action)
        else:
            raise ValueError(f"Unknown policy: {policy_name}")

        obs, reward, done, truncated, info = env.step(action)

        final_rec = float(info["final_recommendation"])
        true_funding = float(info["true_funding"])

        total_reward += reward
        total_final_recommendation += final_rec
        total_true_funding += true_funding
        absolute_error += abs(final_rec - true_funding)
        log_absolute_error += abs(np.log1p(final_rec) - np.log1p(true_funding))

        if info["remaining_budget"] < 0:
            overspend_count += 1

        if true_funding > 0 and final_rec < 0.25 * true_funding:
            severe_underfund_count += 1

        multipliers.append(float(info["multiplier"]))

        if done or truncated:
            break

    return {
        "total_reward": total_reward,
        "total_final_recommendation": total_final_recommendation,
        "total_true_funding": total_true_funding,
        "remaining_budget": info["remaining_budget"],
        "mae": absolute_error / len(multipliers),
        "log_mae": log_absolute_error / len(multipliers),
        "overspend_count": overspend_count,
        "severe_underfund_count": severe_underfund_count,
        "average_multiplier": sum(multipliers) / len(multipliers),
        "multiplier_counts": Counter(multipliers),
    }


def summarize(results):
    keys = [
        "total_reward",
        "total_final_recommendation",
        "total_true_funding",
        "remaining_budget",
        "mae",
        "log_mae",
        "overspend_count",
        "severe_underfund_count",
        "average_multiplier",
    ]

    summary = {}

    for key in keys:
        summary[key] = sum(row[key] for row in results) / len(results)

    all_counts = Counter()

    for row in results:
        all_counts.update(row["multiplier_counts"])

    summary["multiplier_counts"] = dict(all_counts)

    return summary


def print_summary(policy_name, summary):
    print("\n" + "=" * 70)
    print(policy_name)
    print("=" * 70)

    for key, value in summary.items():
        if key == "multiplier_counts":
            print("multiplier_counts:")
            for multiplier, count in sorted(value.items()):
                print(f"  {multiplier}: {count}")
        else:
            print(f"{key}: {value:,.4f}")


def evaluate_clean_model_policy(policy_name, model=None, episode_length=200, num_eval_episodes=30):
    results = []

    for seed in range(num_eval_episodes):
        eval_env = CleanBudgetAdjustmentEnv(
            data_path=RL_TRAINING_DATA_PATH,
            episode_length=episode_length,
            random_seed=1000 + seed,
        )

        results.append(
            run_episode(
                env=eval_env,
                policy_name=policy_name,
                model=model,
            )
        )

    return summarize(results)

In [10]:
clean_dqn_env = CleanBudgetAdjustmentEnv(
    data_path=RL_TRAINING_DATA_PATH,
    episode_length=200,
    random_seed=RANDOM_SEED,
)

clean_dqn_model = DQN(
    policy="MlpPolicy",
    env=clean_dqn_env,
    learning_rate=0.0003,
    buffer_size=100_000,
    learning_starts=2_000,
    batch_size=128,
    gamma=0.995,
    train_freq=4,
    target_update_interval=1_000,
    exploration_fraction=0.30,
    exploration_initial_eps=1.0,
    exploration_final_eps=0.03,
    verbose=1,
    seed=RANDOM_SEED,
)

clean_dqn_model.learn(total_timesteps=100_000)

clean_dqn_model.save("clean_dqn_budget_adjustment_model")

print("Clean DQN saved.")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 200      |
|    ep_rew_mean      | -128     |
|    exploration_rate | 0.974    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 3067     |
|    time_elapsed     | 0        |
|    total_timesteps  | 800      |
----------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 200      |
|    ep_rew_mean      | -160     |
|    exploration_rate | 0.948    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 2911     |
|    time_elapsed     | 0        |
|    total_timesteps  | 1600     |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 200      |
|    ep_rew_mean      | -133     |
|    exploration_rate | 0.922    |
| time/               |          |
|    episodes         | 12       |
|    fps              | 1945     |
|    time_elapsed     | 1        |
|    total_timesteps  | 2400     |
| train/              |          |
|    learning_rate    | 0.0003   |
|    loss             | 0.228    |
|    n_updates        | 99       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean    

In [11]:
clean_ppo_env = CleanBudgetAdjustmentEnv(
    data_path=RL_TRAINING_DATA_PATH,
    episode_length=200,
    random_seed=RANDOM_SEED,
)

clean_ppo_model = PPO(
    policy="MlpPolicy",
    env=clean_ppo_env,
    learning_rate=0.0003,
    n_steps=2048,
    batch_size=64,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.03,
    verbose=1,
    seed=RANDOM_SEED,
)

clean_ppo_model.learn(total_timesteps=100_000)

clean_ppo_model.save("clean_ppo_budget_adjustment_model")

print("Clean PPO saved.")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 200      |
|    ep_rew_mean     | -150     |
| time/              |          |
|    fps             | 935      |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 200         |
|    ep_rew_mean          | -192        |
| time/                   |             |
|    fps                  | 753         |
|    iterations           | 2           |
|    time_elapsed         | 5           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.010820674 |
|    clip_fraction        | 0.0255      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.79       |
|    explained_variance   | 0.0124      |
|    learning_rate        | 0.

In [12]:
clean_a2c_env = CleanBudgetAdjustmentEnv(
    data_path=RL_TRAINING_DATA_PATH,
    episode_length=200,
    random_seed=RANDOM_SEED,
)

clean_a2c_model = A2C(
    policy="MlpPolicy",
    env=clean_a2c_env,
    learning_rate=0.0007,
    gamma=0.99,
    ent_coef=0.01,
    verbose=1,
    seed=RANDOM_SEED,
)

clean_a2c_model.learn(total_timesteps=100_000)

clean_a2c_model.save("clean_a2c_budget_adjustment_model")

print("Clean A2C saved.")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


------------------------------------
| rollout/              |          |
|    ep_len_mean        | 200      |
|    ep_rew_mean        | -273     |
| time/                 |          |
|    fps                | 308      |
|    iterations         | 100      |
|    time_elapsed       | 1        |
|    total_timesteps    | 500      |
| train/                |          |
|    entropy_loss       | -1.78    |
|    explained_variance | 0.216    |
|    learning_rate      | 0.0007   |
|    n_updates          | 99       |
|    policy_loss        | -0.836   |
|    value_loss         | 0.28     |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 200      |
|    ep_rew_mean        | -156     |
| time/                 |          |
|    fps                | 330      |
|    iterations         | 200      |
|    time_elapsed       | 3        |
|    total_timesteps    | 1000     |
| train/                |          |
|

In [13]:
clean_results = {}

clean_results["random"] = evaluate_clean_model_policy("random", model=None)
clean_results["ml_only"] = evaluate_clean_model_policy("ml_only", model=None)
clean_results["rule_based"] = evaluate_clean_model_policy("rule_based", model=None)
clean_results["clean_dqn"] = evaluate_clean_model_policy("dqn", model=clean_dqn_model)
clean_results["clean_ppo"] = evaluate_clean_model_policy("ppo", model=clean_ppo_model)
clean_results["clean_a2c"] = evaluate_clean_model_policy("a2c", model=clean_a2c_model)

for policy_name, summary in clean_results.items():
    print_summary(policy_name, summary)


random
total_reward: -265.8030
total_final_recommendation: 1,735,045,145.9627
total_true_funding: 4,562,377,748.8870
remaining_budget: 964,954,854.0373
mae: 14,882,410.1323
log_mae: 1.1224
overspend_count: 0.0000
severe_underfund_count: 17.2333
average_multiplier: 0.7714
multiplier_counts:
  0.4: 961
  0.55: 1022
  0.7: 958
  0.85: 999
  1.0: 1040
  1.1: 1020

ml_only
total_reward: -319.5844
total_final_recommendation: 1,910,173,223.9990
total_true_funding: 4,562,377,748.8870
remaining_budget: 789,826,776.0010
mae: 14,302,508.0112
log_mae: 1.3336
overspend_count: 0.0000
severe_underfund_count: 20.0000
average_multiplier: 1.0000
multiplier_counts:
  1.0: 6000

rule_based
total_reward: -269.6266
total_final_recommendation: 1,781,764,640.8372
total_true_funding: 4,562,377,748.8870
remaining_budget: 918,235,359.1628
mae: 14,586,678.0759
log_mae: 1.1311
overspend_count: 0.0000
severe_underfund_count: 16.6667
average_multiplier: 0.7785
multiplier_counts:
  0.55: 1459
  0.7: 1434
  0.85: 161

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [14]:
def evaluate_clean_model_policy(policy_name, model=None, episode_length=200, num_eval_episodes=30):
    results = []

    for seed in range(num_eval_episodes):
        eval_env = CleanBudgetAdjustmentEnv(
            data_path=RL_TRAINING_DATA_PATH,
            episode_length=episode_length,
            random_seed=1000 + seed,
        )

        results.append(
            run_episode(
                env=eval_env,
                policy_name=policy_name,
                model=model,
            )
        )

    return summarize(results)

In [15]:
def evaluate_clean_model_policy(policy_name, model=None, episode_length=200, num_eval_episodes=30):
    results = []

    for seed in range(num_eval_episodes):
        eval_env = CleanBudgetAdjustmentEnv(
            data_path=RL_TRAINING_DATA_PATH,
            episode_length=episode_length,
            random_seed=1000 + seed,
        )

        results.append(
            run_episode(
                env=eval_env,
                policy_name=policy_name,
                model=model,
            )
        )

    return summarize(results)

In [16]:
clean_a2c_tuning_results = []
best_clean_a2c_model = None
best_clean_a2c_score = -999999
best_clean_a2c_name = None

for params in clean_a2c_experiments:
    print("\n" + "=" * 80)
    print("Training:", params["name"])
    print("=" * 80)

    env = CleanBudgetAdjustmentEnv(
        data_path=RL_TRAINING_DATA_PATH,
        episode_length=200,
        random_seed=RANDOM_SEED,
    )

    model = A2C(
        policy="MlpPolicy",
        env=env,
        learning_rate=params["learning_rate"],
        gamma=params["gamma"],
        ent_coef=params["ent_coef"],
        verbose=0,
        seed=RANDOM_SEED,
    )

    model.learn(total_timesteps=100_000)

    summary = evaluate_clean_model_policy(
        policy_name="a2c",
        model=model,
        episode_length=200,
        num_eval_episodes=30,
    )

    row = {
        "model": params["name"],
        "algorithm": "A2C",
        "total_reward": summary["total_reward"],
        "mae": summary["mae"],
        "log_mae": summary["log_mae"],
        "remaining_budget": summary["remaining_budget"],
        "overspend_count": summary["overspend_count"],
        "severe_underfund_count": summary["severe_underfund_count"],
        "average_multiplier": summary["average_multiplier"],
        "params": params,
    }

    clean_a2c_tuning_results.append(row)

    print_summary(params["name"], summary)

    if summary["total_reward"] > best_clean_a2c_score:
        best_clean_a2c_score = summary["total_reward"]
        best_clean_a2c_model = model
        best_clean_a2c_name = params["name"]

print("Best Clean A2C:", best_clean_a2c_name, best_clean_a2c_score)

best_clean_a2c_model.save("best_clean_a2c_tuned_model")

NameError: name 'clean_a2c_experiments' is not defined

In [17]:
clean_a2c_experiments = [
    {
        "name": "clean_a2c_base",
        "learning_rate": 0.0007,
        "gamma": 0.99,
        "ent_coef": 0.01,
    },
    {
        "name": "clean_a2c_lower_lr",
        "learning_rate": 0.0003,
        "gamma": 0.99,
        "ent_coef": 0.01,
    },
    {
        "name": "clean_a2c_very_low_lr",
        "learning_rate": 0.0001,
        "gamma": 0.99,
        "ent_coef": 0.01,
    },
    {
        "name": "clean_a2c_more_exploration",
        "learning_rate": 0.0007,
        "gamma": 0.99,
        "ent_coef": 0.03,
    },
    {
        "name": "clean_a2c_less_exploration",
        "learning_rate": 0.0007,
        "gamma": 0.99,
        "ent_coef": 0.001,
    },
    {
        "name": "clean_a2c_long_term",
        "learning_rate": 0.0003,
        "gamma": 0.995,
        "ent_coef": 0.01,
    },
]

In [18]:
clean_a2c_tuning_results = []
best_clean_a2c_model = None
best_clean_a2c_score = -999999
best_clean_a2c_name = None

for params in clean_a2c_experiments:
    print("\n" + "=" * 80)
    print("Training:", params["name"])
    print("=" * 80)

    env = CleanBudgetAdjustmentEnv(
        data_path=RL_TRAINING_DATA_PATH,
        episode_length=200,
        random_seed=RANDOM_SEED,
    )

    model = A2C(
        policy="MlpPolicy",
        env=env,
        learning_rate=params["learning_rate"],
        gamma=params["gamma"],
        ent_coef=params["ent_coef"],
        verbose=0,
        seed=RANDOM_SEED,
    )

    model.learn(total_timesteps=100_000)

    summary = evaluate_clean_model_policy(
        policy_name="a2c",
        model=model,
        episode_length=200,
        num_eval_episodes=30,
    )

    row = {
        "model": params["name"],
        "algorithm": "A2C",
        "total_reward": summary["total_reward"],
        "mae": summary["mae"],
        "log_mae": summary["log_mae"],
        "remaining_budget": summary["remaining_budget"],
        "overspend_count": summary["overspend_count"],
        "severe_underfund_count": summary["severe_underfund_count"],
        "average_multiplier": summary["average_multiplier"],
        "params": params,
    }

    clean_a2c_tuning_results.append(row)

    print_summary(params["name"], summary)

    if summary["total_reward"] > best_clean_a2c_score:
        best_clean_a2c_score = summary["total_reward"]
        best_clean_a2c_model = model
        best_clean_a2c_name = params["name"]

print("Best Clean A2C:", best_clean_a2c_name, best_clean_a2c_score)

best_clean_a2c_model.save("best_clean_a2c_tuned_model")


Training: clean_a2c_base


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



clean_a2c_base
total_reward: -193.9318
total_final_recommendation: 1,603,404,686.8160
total_true_funding: 4,562,377,748.8870
remaining_budget: 1,096,595,313.1840
mae: 15,360,016.9200
log_mae: 0.8262
overspend_count: 0.0000
severe_underfund_count: 11.9000
average_multiplier: 0.8251
multiplier_counts:
  0.4: 331
  0.7: 2
  0.85: 5667

Training: clean_a2c_lower_lr


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



clean_a2c_lower_lr
total_reward: -274.2486
total_final_recommendation: 1,801,846,775.2825
total_true_funding: 4,562,377,748.8870
remaining_budget: 898,153,224.7175
mae: 14,353,776.1349
log_mae: 1.1572
overspend_count: 0.0000
severe_underfund_count: 17.0667
average_multiplier: 0.8176
multiplier_counts:
  0.4: 432
  0.85: 5567
  1.0: 1

Training: clean_a2c_very_low_lr


...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



clean_a2c_less_exploration
total_reward: -274.0788
total_final_recommendation: 1,801,945,097.8454
total_true_funding: 4,562,377,748.8870
remaining_budget: 898,054,902.1546
mae: 14,353,298.7756
log_mae: 1.1566
overspend_count: 0.0000
severe_underfund_count: 17.0333
average_multiplier: 0.8607
multiplier_counts:
  0.7: 3
  0.85: 5566
  1.0: 431

Training: clean_a2c_long_term


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



clean_a2c_long_term
total_reward: -274.2486
total_final_recommendation: 1,801,846,775.2825
total_true_funding: 4,562,377,748.8870
remaining_budget: 898,153,224.7175
mae: 14,353,776.1349
log_mae: 1.1572
overspend_count: 0.0000
severe_underfund_count: 17.0667
average_multiplier: 0.8176
multiplier_counts:
  0.4: 433
  0.85: 5565
  1.1: 2
Best Clean A2C: clean_a2c_base -193.93179849038415


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [19]:
clean_dqn_experiments = [
    {
        "name": "clean_dqn_base",
        "learning_rate": 0.0003,
        "buffer_size": 100_000,
        "learning_starts": 2_000,
        "batch_size": 128,
        "gamma": 0.995,
        "exploration_fraction": 0.30,
        "exploration_final_eps": 0.03,
    },
    {
        "name": "clean_dqn_lower_lr",
        "learning_rate": 0.0001,
        "buffer_size": 100_000,
        "learning_starts": 2_000,
        "batch_size": 128,
        "gamma": 0.995,
        "exploration_fraction": 0.30,
        "exploration_final_eps": 0.03,
    },
    {
        "name": "clean_dqn_more_exploration",
        "learning_rate": 0.0003,
        "buffer_size": 100_000,
        "learning_starts": 2_000,
        "batch_size": 128,
        "gamma": 0.995,
        "exploration_fraction": 0.45,
        "exploration_final_eps": 0.05,
    },
    {
        "name": "clean_dqn_less_exploration",
        "learning_rate": 0.0003,
        "buffer_size": 100_000,
        "learning_starts": 2_000,
        "batch_size": 128,
        "gamma": 0.995,
        "exploration_fraction": 0.20,
        "exploration_final_eps": 0.01,
    },
    {
        "name": "clean_dqn_smaller_batch",
        "learning_rate": 0.0003,
        "buffer_size": 100_000,
        "learning_starts": 1_000,
        "batch_size": 64,
        "gamma": 0.99,
        "exploration_fraction": 0.30,
        "exploration_final_eps": 0.03,
    },
]

In [20]:
clean_dqn_tuning_results = []
best_clean_dqn_model = None
best_clean_dqn_score = -999999
best_clean_dqn_name = None

for params in clean_dqn_experiments:
    print("\n" + "=" * 80)
    print("Training:", params["name"])
    print("=" * 80)

    env = CleanBudgetAdjustmentEnv(
        data_path=RL_TRAINING_DATA_PATH,
        episode_length=200,
        random_seed=RANDOM_SEED,
    )

    model = DQN(
        policy="MlpPolicy",
        env=env,
        learning_rate=params["learning_rate"],
        buffer_size=params["buffer_size"],
        learning_starts=params["learning_starts"],
        batch_size=params["batch_size"],
        gamma=params["gamma"],
        train_freq=4,
        target_update_interval=1_000,
        exploration_fraction=params["exploration_fraction"],
        exploration_initial_eps=1.0,
        exploration_final_eps=params["exploration_final_eps"],
        verbose=0,
        seed=RANDOM_SEED,
    )

    model.learn(total_timesteps=100_000)

    summary = evaluate_clean_model_policy(
        policy_name="dqn",
        model=model,
        episode_length=200,
        num_eval_episodes=30,
    )

    row = {
        "model": params["name"],
        "algorithm": "DQN",
        "total_reward": summary["total_reward"],
        "mae": summary["mae"],
        "log_mae": summary["log_mae"],
        "remaining_budget": summary["remaining_budget"],
        "overspend_count": summary["overspend_count"],
        "severe_underfund_count": summary["severe_underfund_count"],
        "average_multiplier": summary["average_multiplier"],
        "params": params,
    }

    clean_dqn_tuning_results.append(row)

    print_summary(params["name"], summary)

    if summary["total_reward"] > best_clean_dqn_score:
        best_clean_dqn_score = summary["total_reward"]
        best_clean_dqn_model = model
        best_clean_dqn_name = params["name"]

print("Best Clean DQN:", best_clean_dqn_name, best_clean_dqn_score)

best_clean_dqn_model.save("best_clean_dqn_tuned_model")


Training: clean_dqn_base


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



clean_dqn_base
total_reward: -196.6562
total_final_recommendation: 1,640,906,165.3104
total_true_funding: 4,562,377,748.8870
remaining_budget: 1,059,093,834.6896
mae: 15,488,987.8091
log_mae: 0.8386
overspend_count: 0.0000
severe_underfund_count: 12.0667
average_multiplier: 0.8328
multiplier_counts:
  0.4: 76
  0.55: 400
  0.7: 2549
  0.85: 630
  1.0: 1531
  1.1: 814

Training: clean_dqn_lower_lr


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



clean_dqn_lower_lr
total_reward: -198.7262
total_final_recommendation: 1,568,600,425.5919
total_true_funding: 4,562,377,748.8870
remaining_budget: 1,131,399,574.4081
mae: 15,531,733.1385
log_mae: 0.8535
overspend_count: 0.0000
severe_underfund_count: 11.6000
average_multiplier: 0.7651
multiplier_counts:
  0.4: 276
  0.55: 3055
  0.7: 340
  1.0: 2
  1.1: 2327

Training: clean_dqn_more_exploration


...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



clean_dqn_less_exploration
total_reward: -191.9704
total_final_recommendation: 1,628,453,692.7273
total_true_funding: 4,562,377,748.8870
remaining_budget: 1,071,546,307.2727
mae: 15,459,079.7722
log_mae: 0.8164
overspend_count: 0.0000
severe_underfund_count: 11.9000
average_multiplier: 0.5958
multiplier_counts:
  0.4: 3798
  0.85: 976
  1.0: 1226

Training: clean_dqn_smaller_batch


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



clean_dqn_smaller_batch
total_reward: -192.9313
total_final_recommendation: 1,645,147,568.6253
total_true_funding: 4,562,377,748.8870
remaining_budget: 1,054,852,431.3747
mae: 15,376,597.1135
log_mae: 0.8210
overspend_count: 0.0000
severe_underfund_count: 11.9333
average_multiplier: 0.8225
multiplier_counts:
  0.4: 707
  0.7: 1840
  0.85: 594
  1.0: 2859
Best Clean DQN: clean_dqn_more_exploration -191.87378930347273


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [21]:
clean_ppo_experiments = [
    {
        "name": "clean_ppo_base",
        "learning_rate": 0.0003,
        "n_steps": 2048,
        "batch_size": 64,
        "gamma": 0.99,
        "gae_lambda": 0.95,
        "clip_range": 0.2,
        "ent_coef": 0.03,
    },
    {
        "name": "clean_ppo_lower_lr",
        "learning_rate": 0.0001,
        "n_steps": 2048,
        "batch_size": 64,
        "gamma": 0.99,
        "gae_lambda": 0.95,
        "clip_range": 0.2,
        "ent_coef": 0.03,
    },
    {
        "name": "clean_ppo_less_exploration",
        "learning_rate": 0.0003,
        "n_steps": 2048,
        "batch_size": 64,
        "gamma": 0.99,
        "gae_lambda": 0.95,
        "clip_range": 0.2,
        "ent_coef": 0.001,
    },
    {
        "name": "clean_ppo_more_exploration",
        "learning_rate": 0.0003,
        "n_steps": 2048,
        "batch_size": 64,
        "gamma": 0.99,
        "gae_lambda": 0.95,
        "clip_range": 0.2,
        "ent_coef": 0.05,
    },
    {
        "name": "clean_ppo_longer_memory",
        "learning_rate": 0.0003,
        "n_steps": 4096,
        "batch_size": 128,
        "gamma": 0.995,
        "gae_lambda": 0.95,
        "clip_range": 0.2,
        "ent_coef": 0.03,
    },
]

In [22]:
clean_ppo_tuning_results = []
best_clean_ppo_model = None
best_clean_ppo_score = -999999
best_clean_ppo_name = None

for params in clean_ppo_experiments:
    print("\n" + "=" * 80)
    print("Training:", params["name"])
    print("=" * 80)

    env = CleanBudgetAdjustmentEnv(
        data_path=RL_TRAINING_DATA_PATH,
        episode_length=200,
        random_seed=RANDOM_SEED,
    )

    model = PPO(
        policy="MlpPolicy",
        env=env,
        learning_rate=params["learning_rate"],
        n_steps=params["n_steps"],
        batch_size=params["batch_size"],
        gamma=params["gamma"],
        gae_lambda=params["gae_lambda"],
        clip_range=params["clip_range"],
        ent_coef=params["ent_coef"],
        verbose=0,
        seed=RANDOM_SEED,
    )

    model.learn(total_timesteps=100_000)

    summary = evaluate_clean_model_policy(
        policy_name="ppo",
        model=model,
        episode_length=200,
        num_eval_episodes=30,
    )

    row = {
        "model": params["name"],
        "algorithm": "PPO",
        "total_reward": summary["total_reward"],
        "mae": summary["mae"],
        "log_mae": summary["log_mae"],
        "remaining_budget": summary["remaining_budget"],
        "overspend_count": summary["overspend_count"],
        "severe_underfund_count": summary["severe_underfund_count"],
        "average_multiplier": summary["average_multiplier"],
        "params": params,
    }

    clean_ppo_tuning_results.append(row)

    print_summary(params["name"], summary)

    if summary["total_reward"] > best_clean_ppo_score:
        best_clean_ppo_score = summary["total_reward"]
        best_clean_ppo_model = model
        best_clean_ppo_name = params["name"]

print("Best Clean PPO:", best_clean_ppo_name, best_clean_ppo_score)

best_clean_ppo_model.save("best_clean_ppo_tuned_model")


Training: clean_ppo_base


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



clean_ppo_base
total_reward: -228.4121
total_final_recommendation: 1,386,519,869.2289
total_true_funding: 4,562,377,748.8870
remaining_budget: 1,313,480,130.7711
mae: 16,152,471.5930
log_mae: 0.9618
overspend_count: 0.0000
severe_underfund_count: 17.2333
average_multiplier: 0.5897
multiplier_counts:
  0.4: 3867
  0.7: 887
  1.1: 1246

Training: clean_ppo_lower_lr


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



clean_ppo_lower_lr
total_reward: -274.6566
total_final_recommendation: 1,805,937,572.3827
total_true_funding: 4,562,377,748.8870
remaining_budget: 894,062,427.6173
mae: 14,396,020.2543
log_mae: 1.1568
overspend_count: 0.0000
severe_underfund_count: 17.0000
average_multiplier: 0.8646
multiplier_counts:
  0.85: 5650
  1.1: 350

Training: clean_ppo_less_exploration


...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



clean_ppo_more_exploration
total_reward: -207.7188
total_final_recommendation: 1,499,305,726.7881
total_true_funding: 4,562,377,748.8870
remaining_budget: 1,200,694,273.2119
mae: 15,888,551.7790
log_mae: 0.8845
overspend_count: 0.0000
severe_underfund_count: 13.8000
average_multiplier: 0.6671
multiplier_counts:
  0.4: 2954
  0.55: 304
  0.7: 523
  1.0: 1532
  1.1: 687

Training: clean_ppo_longer_memory


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



clean_ppo_longer_memory
total_reward: -194.7144
total_final_recommendation: 1,694,925,286.4210
total_true_funding: 4,562,377,748.8870
remaining_budget: 1,005,074,713.5790
mae: 15,481,303.4236
log_mae: 0.8352
overspend_count: 0.0000
severe_underfund_count: 11.6000
average_multiplier: 0.9211
multiplier_counts:
  0.4: 390
  0.7: 1581
  0.85: 673
  1.1: 3356
Best Clean PPO: clean_ppo_longer_memory -194.71437203748536


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [23]:
all_clean_tuning_results = (
    clean_a2c_tuning_results
    + clean_dqn_tuning_results
    + clean_ppo_tuning_results
)

clean_tuning_df = pd.DataFrame(all_clean_tuning_results)

clean_tuning_df_clean = clean_tuning_df.drop(columns=["params"])
clean_tuning_df_clean = clean_tuning_df_clean.sort_values("total_reward", ascending=False)

clean_tuning_df_clean

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,model,algorithm,total_reward,mae,log_mae,remaining_budget,overspend_count,severe_underfund_count,average_multiplier
8,clean_dqn_more_exploration,DQN,-191.873789,1.544109e+07,0.816462,1.059250e+09,0.0,11.833333,0.827450
9,clean_dqn_less_exploration,DQN,-191.970396,1.545908e+07,0.816439,1.071546e+09,0.0,11.900000,0.595800
10,clean_dqn_smaller_batch,DQN,-192.931327,1.537660e+07,0.821002,1.054852e+09,0.0,11.933333,0.822450
0,clean_a2c_base,A2C,-193.931798,1.536002e+07,0.826235,1.096595e+09,0.0,11.900000,0.825125
15,clean_ppo_longer_memory,PPO,-194.714372,1.548130e+07,0.835190,1.005075e+09,0.0,11.600000,0.921058
6,clean_dqn_base,DQN,-196.656156,1.548899e+07,0.838624,1.059094e+09,0.0,12.066667,0.832767
7,clean_dqn_lower_lr,DQN,-198.726174,1.553173e+07,0.853524,1.131400e+09,0.0,11.600000,0.765058
13,clean_ppo_less_exploration,PPO,-201.370335,1.538306e+07,0.856987,9.675945e+08,0.0,12.200000,0.915417
14,clean_ppo_more_exploration,PPO,-207.718819,1.588855e+07,0.884455,1.200694e+09,0.0,13.800000,0.667100
3,clean_a2c_more_exploration,A2C,-224.410371,1.461062e+07,0.945778,9.509962e+08,0.0,13.566667,0.822725


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [24]:
final_clean_results = {}

final_clean_results["random"] = evaluate_clean_model_policy("random", model=None)
final_clean_results["ml_only"] = evaluate_clean_model_policy("ml_only", model=None)
final_clean_results["rule_based"] = evaluate_clean_model_policy("rule_based", model=None)
final_clean_results["best_clean_a2c"] = evaluate_clean_model_policy("a2c", model=best_clean_a2c_model)
final_clean_results["best_clean_dqn"] = evaluate_clean_model_policy("dqn", model=best_clean_dqn_model)
final_clean_results["best_clean_ppo"] = evaluate_clean_model_policy("ppo", model=best_clean_ppo_model)

final_clean_rows = []

for name, summary in final_clean_results.items():
    final_clean_rows.append({
        "policy": name,
        "total_reward": summary["total_reward"],
        "mae": summary["mae"],
        "log_mae": summary["log_mae"],
        "remaining_budget": summary["remaining_budget"],
        "overspend_count": summary["overspend_count"],
        "severe_underfund_count": summary["severe_underfund_count"],
        "average_multiplier": summary["average_multiplier"],
    })

final_clean_comparison_df = pd.DataFrame(final_clean_rows)
final_clean_comparison_df = final_clean_comparison_df.sort_values("total_reward", ascending=False)

final_clean_comparison_df

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,policy,total_reward,mae,log_mae,remaining_budget,overspend_count,severe_underfund_count,average_multiplier
4,best_clean_dqn,-191.873789,1.544109e+07,0.816462,1.059250e+09,0.0,11.833333,0.827450
3,best_clean_a2c,-193.931798,1.536002e+07,0.826235,1.096595e+09,0.0,11.900000,0.825125
5,best_clean_ppo,-194.714372,1.548130e+07,0.835190,1.005075e+09,0.0,11.600000,0.921058
0,random,-261.245151,1.500482e+07,1.104095,1.002526e+09,0.0,16.800000,0.764125
2,rule_based,-269.626632,1.458668e+07,1.131125,9.182354e+08,0.0,16.666667,0.778500
1,ml_only,-319.584355,1.430251e+07,1.333639,7.898268e+08,0.0,20.000000,1.000000


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [25]:
clean_tuning_df_clean.to_csv("clean_rl_tuning_results.csv", index=False)
final_clean_comparison_df.to_csv("final_clean_tuned_rl_comparison.csv", index=False)

files.download("clean_rl_tuning_results.csv")
files.download("final_clean_tuned_rl_comparison.csv")

files.download("best_clean_dqn_tuned_model.zip")
files.download("best_clean_a2c_tuned_model.zip")
files.download("best_clean_ppo_tuned_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>